# Abstract

# Dataset Description

### General Description

The Online Gaming Anxiety Dataset is a large-scale public dataset compiled by researchers __Marian Sauter__ and __Dejan Draschkow__ (2017). It was designed to empirically investigate the relationship between online video gaming behaviors (e.g., playtimes, social contexts, motivations) and mental health indicators—specifically generalized anxiety, social phobia, and overall life satisfaction.

### How the Dataset was Collected

- __Method:__ The researchers used an _online survey/questionnaire_ hosted on the Open Science Framework (OSF)

- __Recruitment:__ Convenience sampling was employed by posting invitations containing the links to the survey across online gaming communities and forums such as `r/gaming`, `r/leagueoflegends`.

### Potential Implications of Data Collection on Generated Insights

- __Self-Selection Bias:__ Due to the voluntary nature of the survey, individuals with strong opinions about gaming or those experiencing higher levels of psychological distress may have been more motivated to participate.

- __Community Bias (Hardcore vs Casual):__ There may be an underrepresentation of _Casual, Offline, or mobile-first_ gamers due to the survey recruitment being posted in dedicated gaming forums. This may show a bias towards highly active, competitive, and "hardcore" gamers.

- __Self-Reporting and Recall Bias:__ Metrics such as playtime(`Hours`) and stream watchtime(`streams`) are self-reported which could lead to estimation errors. Psychological scores based on honest self-assessment are vulnerable to social desirability bias.

- __Lack of a Control Group:__ Due to the primary survey targets being active gamers, there is no baseline for mental health statistics of non-gamers for comparison.

### Structure of the Data

- __Representation__
> - __Rows:__ Each row represents a single, anonymous response.
> - __Columns:__ Each column respresents a specific attribute including demographics, gaming behaviors, or an aggregated clinical scale.

- __Number of Observations__
> - __Original Dataset:__ Contains `13,464` entries across `55` columns.
> - __Cleaned Dataset:__ Contains `12,562` entries across `19` standardized columns after dropping metadata, individual questionnaire items, null values, and outliers with weekly playtime `>80 hours`.

### Attribute Dictionary

1. __Demographic Attributes__

> - `Age:` __Numeric__, The age of the respondent _(18 to 56 years)_.
> - `Gender:` __Categorical__, Respondent's chosen gender identity _(Male, Female, Other)_.
> - `Degree:` __Categorical__, Highest educational attainment _(High School, Bachelor's, Master's, Ph.D.)_.
> - `Work:` __Categorical__, Employment Status _(Employed, Unemployed, Student)_.
> - `Birthplace` & `Residence:` __Categorical__, Countries of Birth and Current residence.

2. __Gaming Behavior & Motivation Attributes__

> - `Hours:` __Numeric__, Average weekly hours spent playing video games _(Max of 80 hrs)_.
> - `streams:` __Numeric__, Average weekly hours spent watching streams of video games.
> - `Platform:` __Categorical__, Primary system used for gaming _(PC, Console, Mobile)__.
> - `Playstyle:` __Categorical__, Social setting of play _(Singleplayer, Multiplayer with online friends, offline friends, or strangers)_.
> - `League_Standardized:` __Categorical__, Standardized competitive rank/tier _(Master+, Diamond, Platinum, Gold, Silver, Bronze, Unranked)_.
> - `whyplay_Standardized:` __Categorical__, Standardized primary reason for playing _(Having Fun, Improving, Winning, Relaxing, Escapism, Mixed, Other)_.
> - `earnings:` __Categorical__, Whether they play purely for fun or for financial compensation.
> - `Game:` __Categorical__, Respondent's chosen primary game _(Counter-Strike, World of Warcraft, Skyrim, League of Legends, Other)_.

3. __Clinical & Psychological Attributes (Aggregated)__

> - `GAD_T:` __Numeric__, Total score from the __GAD-7 (Generalized Anxiety Disorder)__ scale, measuring anxiety severity over the past 2 weeks with a range of `0-21` _(Higher = higher anxiety level)_.
> - `SWL_T:` __Numeric__, Total score from the __SWLS (Satisfaction With Life Scale)__, measuring subjective life satisfaction with a range of `5-35` _(Higher = greater satisfaction)_.
> - `SPIN_T:` __Numeric__, Total score from the __SPIN (Social Phobia Inventory)__, measuring social anxiety sympotoms with a range of `0-68` _(Higher = more severe social anxiety)_.
> - `GADE:` __Categorical__, An optional GAD assessment question indicating how difficult their anxiety makes daily life/work.
> - `Narcissism:` __Numeric__, Self-reported single-item narcissism scale score with a range of `1-5`.

## Import required modules and download csv file.

In [661]:
import kagglehub
import polars as pl
import os
path = kagglehub.dataset_download("divyansh22/online-gaming-anxiety-data")

print("Path to dataset files:", path)
csv = os.path.join(path, "GamingStudy_data.csv")
df = pl.read_csv(csv, encoding="latin1", null_values="NA")

Path to dataset files: C:\Users\jlgfd\.cache\kagglehub\datasets\divyansh22\online-gaming-anxiety-data\versions\3


# Data Cleaning and Preprocessing

## Drop Unnecessary Columns
These columns were dropped as this study will utilize the aggregate variables rather than individual questionnaire items:
- `S. No.` & `Timestamp` (not useful for statistical modeling)
- `GAD1` to `GAD7` (redundant, represented by the total score `GAD_T`)
- `SWL1` to `SWL5` (redundant, represented by `SWL_T`)
- `SPIN1` to `SPIN17` (redundant, represented by `SPIN_T`)
- `highestleague` (100% null/missing values)
- `Residence_ISO3` & `Birthplace_ISO3` (redundant, represented by `Residence` and `Birthplace` respectively)
- `accept` (participant consent column, contains no predictive information for modeling)
- `reference` (contains no predictive information for modeling)

In [ ]:
dropped_columns = [
    "S. No.", "Timestamp", "GAD1", "GAD2", "GAD3", "GAD4", "GAD5", "GAD6", "GAD7", 
    "SWL1", "SWL2", "SWL3", "SWL4", "SWL5", "SPIN1", "SPIN2", "SPIN3", "SPIN4", 
    "SPIN5", "SPIN6", "SPIN7", "SPIN8", "SPIN9", "SPIN10", "SPIN11", "SPIN12", 
    "SPIN13", "SPIN14", "SPIN15", "SPIN16", "SPIN17", "highestleague",
    "Residence_ISO3", "Birthplace_ISO3", "accept", "Reference"
]

df_clean = df.drop(dropped_columns)
df_clean.head()

GADE,Game,Platform,Hours,earnings,whyplay,League,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Playstyle,GAD_T,SWL_T,SPIN_T
str,str,str,i64,str,str,str,i64,i64,str,i64,str,str,str,str,str,i64,i64,i64
"""Not difficult at all""","""Skyrim""","""Console (PS, Xbox, ...)""",15,"""I play for fun""","""having fun""","""N/A""",0,1,"""Male""",25,"""Unemployed / between jobs""","""Bachelor (or equivalent)""","""USA""","""USA""","""Singleplayer""",1,23,5
"""Somewhat difficult""","""Other""","""PC""",8,"""I play for fun""","""having fun""","""N/A""",2,1,"""Male""",41,"""Unemployed / between jobs""","""Bachelor (or equivalent)""","""USA""","""USA""","""Multiplayer - online - with st…",8,16,33
"""Not difficult at all""","""Other""","""PC""",0,"""I play for fun""","""having fun""","""n/a""",0,4,"""Female""",32,"""Employed""","""Bachelor (or equivalent)""","""Germany""","""Germany""","""Singleplayer""",8,17,31
"""Not difficult at all""","""Other""","""PC""",20,"""I play for fun""","""improving""","""N/A""",5,2,"""Male""",28,"""Employed""","""Bachelor (or equivalent)""","""USA""","""USA""","""Multiplayer - online - with on…",0,17,11
"""Very difficult""","""Other""","""Console (PS, Xbox, ...)""",20,"""I play for fun""","""having fun""","""n/a""",1,1,"""Male""",19,"""Employed""","""High school diploma (or equiva…","""USA""","""South Korea""","""Multiplayer - online - with st…",14,14,13


## Outlier Treatment

Drop unrealistic values in the `Hours` column (participants reporting >80 hours of gaming per week).

In [663]:
df_clean["Hours"].describe()

statistic,value
str,f64
"""count""",13434.0
"""null_count""",30.0
"""mean""",22.247357
"""std""",70.284502
"""min""",0.0
"""25%""",12.0
"""50%""",20.0
"""75%""",28.0
"""max""",8000.0


In [664]:
df_clean = df_clean.filter(pl.col("Hours") <= 80)
df_clean.shape

(13382, 19)

# Null Cleaning
Each column was checked for null values. Whether to drop or impute nulls was decided based on the proportion and nature of the missingness.

In [665]:
df_clean.null_count()

GADE,Game,Platform,Hours,earnings,whyplay,League,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Playstyle,GAD_T,SWL_T,SPIN_T
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
645,0,0,0,0,0,1739,89,23,0,0,38,0,0,0,0,0,0,646


### Null Value Imputation and Dropping

1. **GADE (How gaming affects daily life/work)**: The `GADE` question is optional in the official GAD questionnaire and is skipped if the participant has zero anxiety. Therefore, we impute missing `GADE` values as `"Not difficult at all"` if `GAD_T == 0`. The remaining null values for `GADE` (where `GAD_T > 0`) are dropped.
2. **Other Nulls**: We drop rows containing null values for `Narcissism`, `streams`, `Work`, and `SPIN_T` as they represent a small percentage of the dataset.

In [666]:
# Impute GADE as 'Not difficult at all' when GAD_T is 0
df_clean = df_clean.with_columns(
    pl.when(pl.col("GADE").is_null() & (pl.col("GAD_T") == 0))
    .then(pl.lit("Not difficult at all"))
    .otherwise(pl.col("GADE"))
    .alias("GADE")
)

# Drop remaining nulls across key columns
df_clean = df_clean.drop_nulls(["Narcissism", "streams", "Work", "GADE", "SPIN_T"])
df_clean.shape

(12562, 19)

*Take note the "Work" column includes unemployed, students and employment so there's no valid reason to have it as null*

## Standardizing the League and whyplay Columns

Since the `League` and `whyplay` columns are free-text response fields, they contain highly unstandardized entries (custom comments, typos, and mixtures). We standardize them into discrete categories.

#### 1. League Rank Standardization:
- **Master+**: Challenger, Master, Grandmaster, GM, and common typos (*challenjour*, etc.).
- **Diamond**: Diamond, Dia, and abbreviations (*d1* to *d5*).
- **Platinum**: Platinum, Plat, and spelling variations (*plarinum*, *plantinum*).
- **Gold**: Gold and typos (*glod*, *golld*, *goled*).
- **Silver**: Silver and spelling variations (*sliver*, *siver*, *silber*).
- **Bronze**: Bronze and spelling variations (*bronce*, *broze*).
- **Unranked**: Captures all non-league values, including blank/unranked responses, CS:GO ranks, and noise.

#### 2. whyplay (Reason for Playing) Standardization:
- **Having Fun**: Joy, pleasure, or fun-related motivations.
- **Improving**: Skill progression, learning, and getting better.
- **Winning**: Competition, victory, and rank-climbing.
- **Relaxing**: De-stressing, unwinding, and casual play.
- **Mixed / All**: Custom responses specifying multiple reasons.
- **Escapism**: Passing time, distraction, and escaping reality.
- **Other**: Custom answers.

In [667]:
df_clean = df_clean.with_columns(pl.col("whyplay").str.to_lowercase())
df_clean = df_clean.with_columns(pl.col("League").str.to_lowercase())

In [668]:
# Apply simplified Polars mapping expressions
league_lower = pl.col("League").fill_null("unranked").str.to_lowercase()

df_clean = df_clean.with_columns(
    pl.when(league_lower.str.contains(r"master|grandmaster|challenger|gm"))
    .then(pl.lit("Master+"))
    .when(league_lower.str.contains(r"diamond|dia|\bd[1-5]\b"))
    .then(pl.lit("Diamond"))
    .when(league_lower.str.contains(r"platinum|plat|\bp[1-5]\b"))
    .then(pl.lit("Platinum"))
    .when(league_lower.str.contains(r"gold|\bg[1-5]\b"))
    .then(pl.lit("Gold"))
    .when(league_lower.str.contains(r"silver|\bs[1-5]\b"))
    .then(pl.lit("Silver"))
    .when(league_lower.str.contains(r"bronze|\bb[1-5]\b"))
    .then(pl.lit("Bronze"))
    .otherwise(pl.lit("Unranked"))
    .alias("League_Standardized")
)

# Standardize whyplay
wp_lower = pl.col("whyplay").fill_null("other").str.to_lowercase()

has_fun = wp_lower.str.contains("fun")
has_improv = wp_lower.str.contains("improv")
has_win = wp_lower.str.contains("win")
has_relax = wp_lower.str.contains("relax")
has_escapism = wp_lower.str.contains("escap|distract|forget|time")

is_mixed = wp_lower.str.contains("all|both|mix") | (
    (has_fun.cast(pl.Int32) + has_improv.cast(pl.Int32) + has_win.cast(pl.Int32) + has_relax.cast(pl.Int32)) > 1
)

df_clean = df_clean.with_columns(
    pl.when(is_mixed)
    .then(pl.lit("Mixed / All"))
    .when(has_fun)
    .then(pl.lit("Having Fun"))
    .when(has_improv)
    .then(pl.lit("Improving"))
    .when(has_win)
    .then(pl.lit("Winning"))
    .when(has_relax)
    .then(pl.lit("Relaxing"))
    .when(has_escapism)
    .then(pl.lit("Escapism"))
    .otherwise(pl.lit("Other"))
    .alias("whyplay_Standardized")
)

print(df_clean.select(["League", "League_Standardized"]).head(5))
print(df_clean.select(["whyplay", "whyplay_Standardized"]).head(5))


shape: (5, 2)
┌────────┬─────────────────────┐
│ League ┆ League_Standardized │
│ ---    ┆ ---                 │
│ str    ┆ str                 │
╞════════╪═════════════════════╡
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
│ n/a    ┆ Unranked            │
└────────┴─────────────────────┘
shape: (5, 2)
┌────────────┬──────────────────────┐
│ whyplay    ┆ whyplay_Standardized │
│ ---        ┆ ---                  │
│ str        ┆ str                  │
╞════════════╪══════════════════════╡
│ having fun ┆ Having Fun           │
│ having fun ┆ Having Fun           │
│ having fun ┆ Having Fun           │
│ improving  ┆ Improving            │
│ having fun ┆ Having Fun           │
└────────────┴──────────────────────┘


In [669]:
# Save original vs standardized column mappings to an editable text file
with pl.Config(tbl_rows=-1, fmt_str_lengths=1000, tbl_width_chars=10000):
    league_mapping = str(df_clean.group_by(["League", "League_Standardized"]).len().sort("len", descending=True))
    whyplay_mapping = str(df_clean.group_by(["whyplay", "whyplay_Standardized"]).len().sort("len", descending=True))

# Drop the original columns as they won't be needed anymore    
df_clean.drop_in_place("League")
df_clean.drop_in_place("whyplay")

with open("standardization_output.txt", "w", encoding="utf-8") as f:
    f.write("=== ORIGINAL LEAGUE vs STANDARDIZED LEAGUE ===\n")
    f.write(league_mapping)
    f.write("\n\n=== ORIGINAL WHYPLAY vs STANDARDIZED WHYPLAY ===\n")
    f.write(whyplay_mapping)

# Save aggregate (overall distribution) of standardized columns to a separate file
with pl.Config(tbl_rows=-1, fmt_str_lengths=1000, tbl_width_chars=10000):
    league_aggregate = str(df_clean["League_Standardized"].value_counts(sort=True))
    whyplay_aggregate = str(df_clean["whyplay_Standardized"].value_counts(sort=True))

with open("standardization_aggregate.txt", "w", encoding="utf-8") as f:
    f.write("=== AGGREGATE STANDARDIZED LEAGUE COUNTS ===\n")
    f.write(league_aggregate)
    f.write("\n\n=== AGGREGATE STANDARDIZED WHYPLAY COUNTS ===\n")
    f.write(whyplay_aggregate)

In [671]:
df_clean.describe()

statistic,GADE,Game,Platform,Hours,earnings,streams,Narcissism,Gender,Age,Work,Degree,Birthplace,Residence,Playstyle,GAD_T,SWL_T,SPIN_T,League_Standardized,whyplay_Standardized
str,str,str,str,f64,str,f64,f64,str,f64,str,str,str,str,str,f64,f64,f64,str,str
"""count""","""12562""","""12562""","""12562""",12562.0,"""12562""",12562.0,12562.0,"""12562""",12562.0,"""12562""","""12562""","""12562""","""12562""","""12562""",12562.0,12562.0,12562.0,"""12562""","""12562"""
"""null_count""","""0""","""0""","""0""",0.0,"""0""",0.0,0.0,"""0""",0.0,"""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,"""0""","""0"""
"""mean""",null,null,null,21.2299,null,10.48583,2.027384,null,20.967521,null,null,null,null,null,5.188664,19.821048,19.806002,null,null
"""std""",null,null,null,12.769674,null,10.209103,1.058987,null,3.299173,null,null,null,null,null,4.6856,7.218174,13.410313,null,null
"""min""","""Extremely difficult""","""Counter Strike""","""Console (PS, Xbox, ...)""",0.0,""" play for fun at the moment, b…",0.0,1.0,"""Female""",18.0,"""Employed""","""Bachelor (or equivalent)""","""Afghanistan""","""Albania""",""" Multiplayer - online - with f…",0.0,5.0,0.0,"""Bronze""","""Escapism"""
"""25%""",null,null,null,12.0,null,4.0,1.0,null,18.0,null,null,null,null,null,2.0,14.0,9.0,null,null
"""50%""",null,null,null,20.0,null,8.0,2.0,null,20.0,null,null,null,null,null,4.0,20.0,17.0,null,null
"""75%""",null,null,null,28.0,null,15.0,3.0,null,22.0,null,null,null,null,null,8.0,26.0,28.0,null,null
"""max""","""Very difficult""","""World of Warcraft""","""Smartphone / Tablet""",80.0,"""working towards getting some f…",168.0,5.0,"""Other""",56.0,"""Unemployed / between jobs""","""Ph.D., Psy. D., MD (or equival…","""Zimbabwe""","""Vietnam""","""with strangers/IRL friends and…",21.0,35.0,68.0,"""Unranked""","""Winning"""


# References
- *[Kaggle Dataset Link](https://www.kaggle.com/datasets/divyansh22/online-gaming-anxiety-data/data)*
- *[Original Paper](https://osf.io/preprints/psyarxiv/mfajz_v1)*
- *[Questionnaire](https://osf.io/vnbxk/files/vyr5f?view_only=4c54da075e164ea2a5329f5669d03c41)*